# Board Meeting RAG — 04: Ask a Question

Ask natural-language questions of the indexed board-meeting corpus. The notebook:

1. Retrieves the top-K most relevant chunks from `board_meetings_index` (filtered
   by quarter/entity if you set those widgets).
2. Prints the hits so you can see what the model is reading.
3. Asks Claude Sonnet 4.6 to answer your question using only those chunks,
   with inline citations like `[SMIB p14]`.

Set the `question` widget, run all, read the answer.

In [ ]:
%pip install --quiet mlflow databricks-vectorsearch
dbutils.library.restartPython()

In [ ]:
from mlflow.deployments import get_deploy_client
from databricks.vector_search.client import VectorSearchClient

CATALOG = "acme_holdings"
INDEX_NAME = f"{CATALOG}.embeddings.board_meetings_index"
ENDPOINT_NAME = "acme_rag_endpoint"
ANSWER_MODEL = "databricks-claude-sonnet-4-6"

dbutils.widgets.text("question", "Which entities made new private equity commitments, and to whom?")
dbutils.widgets.text("quarter", "2026Q1", "Quarter filter (blank = all)")
dbutils.widgets.text("entity", "", "Entity filter (blank = all)")
dbutils.widgets.text("num_results", "8", "Chunks to retrieve")

QUESTION = dbutils.widgets.get("question").strip()
QUARTER = dbutils.widgets.get("quarter").strip()
ENTITY = dbutils.widgets.get("entity").strip()
NUM_RESULTS = int(dbutils.widgets.get("num_results").strip() or "8")

assert QUESTION, "Set the 'question' widget before running."

client = get_deploy_client("databricks")
vsc = VectorSearchClient(disable_notice=True)
idx = vsc.get_index(ENDPOINT_NAME, INDEX_NAME)

## Retrieve relevant chunks

In [ ]:
filters: dict = {}
if QUARTER:
    filters["quarter"] = QUARTER
if ENTITY:
    filters["entity"] = ENTITY

res = idx.similarity_search(
    query_text=QUESTION,
    columns=["chunk_id", "entity", "quarter", "meeting_date", "source_file", "page_number", "section", "content"],
    filters=filters or None,
    num_results=NUM_RESULTS,
)

hits = res.get("result", {}).get("data_array", []) or []
assert hits, f"No chunks matched. Loosen the filters (quarter={QUARTER!r}, entity={ENTITY!r})."

print(f"Retrieved {len(hits)} chunks for: {QUESTION!r}")
print(f"Filters: quarter={QUARTER!r} entity={ENTITY!r}\n")
for h in hits:
    chunk_id, entity, quarter, meeting_date, source_file, page_number, section, content = h[:8]
    loc = f"p{page_number}" if page_number else (section or "")
    snippet = (content or "").replace("\n", " ")[:200]
    print(f"[{entity} {quarter} {loc}] {snippet}…")

## Answer the question (Claude Sonnet 4.6, grounded on retrieved chunks)

In [ ]:
ANSWER_SYSTEM = (
    "You answer questions about institutional investment board meeting materials. "
    "Use ONLY the provided chunks. Cite each fact inline like [SMIB p14] or [CRPTF March 2026]. "
    "If the chunks do not contain enough information to answer, say so plainly. "
    "Preserve specific numbers and named funds/entities exactly as written. No speculation, no market commentary."
)

context_blocks = []
for h in hits:
    chunk_id, entity, quarter, meeting_date, source_file, page_number, section, content = h[:8]
    loc = f"p{page_number}" if page_number else (section or "—")
    context_blocks.append(f"[{entity} {quarter} {loc}] (source: {source_file})\n{content}")
context = "\n\n---\n\n".join(context_blocks)

user_prompt = (
    f"Question: {QUESTION}\n\n"
    f"Answer using only the chunks below. Cite inline like [ENTITY pN] or [ENTITY section].\n\n"
    f"=== CHUNKS ===\n{context}"
)

resp = client.predict(
    endpoint=ANSWER_MODEL,
    inputs={
        "messages": [
            {"role": "system", "content": ANSWER_SYSTEM},
            {"role": "user", "content": user_prompt},
        ],
        "max_tokens": 900,
        "temperature": 0.1,
    },
)
answer = resp["choices"][0]["message"]["content"].strip()

print("=" * 60)
print(f"Q: {QUESTION}")
print("=" * 60)
print(answer)